# Time Series Analysis & ARIMA Forecasting
## India Rainfall & Temperature — A Box-Jenkins Journey

---
> **Reference**: Sharma, M.K. et al. (2020). *Time series analysis on precipitation with missing data using stochastic SARIMA*. MAUSAM, 71(4), 617–624.

### What you will learn
| Step | Topic |
|------|-------|
| 1 | What is a Time Series & its 4 components |
| 2 | Load, reshape, and visualise India rainfall data |
| 3 | Seasonal decomposition |
| 4 | Stationarity — ADF test & differencing |
| 5 | ACF & PACF — reading the order of the model |
| 6 | Box-Jenkins 3-step methodology |
| 7 | ARIMA and SARIMA from scratch with `statsmodels` |
| 8 | Ljung-Box residual diagnostics |
| 9 | Forecast the next 12 months with confidence bands |


---
## 1. What is a Time Series?

A **time series** is a sequence of observations recorded at equally-spaced time intervals — daily stock prices, hourly temperatures, monthly rainfall.

> *Think of it as a movie reel: each frame is an observation, and the order of frames matters.*

### 1.1 Four Components of Any Time Series

```
Observed = Trend  +  Seasonal  +  Cyclical  +  Residual
```

| Component | What it is | India Rainfall Example |
|-----------|-----------|------------------------|
| **Trend** | Long-run upward / downward drift | Gradual change in annual rainfall over decades |
| **Seasonal** | Fixed, repeating pattern within a year | Monsoon every June–September |
| **Cyclical** | Multi-year boom-bust oscillations | ENSO (El Niño) driven multi-year cycles |
| **Residual** | Random noise that remains | Year-to-year unpredictable variation |

### 1.2 Stochastic vs Deterministic

Most real-world time series are **stochastic** — they contain a random component.  
If the *statistics* (mean, variance) **do not change with time** → the series is **stationary**.  
ARIMA models require stationarity. Our job is to transform the raw data until it becomes stationary.


---
## 2. Imports & Style


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Optional: auto-order selection
try:
    from pmdarima import auto_arima
    HAS_PMDARIMA = True
except ImportError:
    HAS_PMDARIMA = False

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('All packages loaded successfully!')

---
## 3. Load & Explore the Data

We have **124 years** (1901–2024) of monthly rainfall (mm) across India.  
The data arrives in **wide format** — one row per year, one column per month.  
We need to reshape it into a **long format** time series with a `DatetimeIndex`.


In [ ]:
# ── Load raw data ──────────────────────────────────────────────────────────
rf_raw  = pd.read_excel('india_avg_rainfall.xlsx')
tmp_raw = pd.read_excel('India_avg_temperature.xlsx')

# Row 0 is a label row; actual data starts from row 1
rf_raw  = rf_raw.iloc[1:].copy().reset_index(drop=True)
tmp_raw = tmp_raw.iloc[1:].copy().reset_index(drop=True)

rf_raw['Year']  = rf_raw['Year'].astype(int)
tmp_raw['Year'] = tmp_raw['Year'].astype(int)

print(f"Rainfall data  : {rf_raw.shape[0]} years  |  {rf_raw['Year'].min()}–{rf_raw['Year'].max()}")
print(f"Temperature data: {tmp_raw.shape[0]} years  |  {tmp_raw['Year'].min()}–{tmp_raw['Year'].max()}")
rf_raw[['Year'] + MONTHS].head()

In [ ]:
def wide_to_monthly_ts(df, value_col_name='value'):
    """Reshape wide-format yearly data into a monthly DatetimeIndex Series."""
    long = df[['Year'] + MONTHS].melt(id_vars='Year', var_name='Month', value_name=value_col_name)
    long['Month_num'] = long['Month'].map({m: i+1 for i, m in enumerate(MONTHS)})
    long['Date'] = pd.to_datetime(dict(year=long['Year'], month=long['Month_num'], day=1))
    return long.set_index('Date').sort_index()[value_col_name].astype(float)

rain_ts = wide_to_monthly_ts(rf_raw,  'rainfall_mm')
temp_ts = wide_to_monthly_ts(tmp_raw, 'temp_C')

print(f"Monthly rainfall series  : {len(rain_ts)} observations  ({rain_ts.index[0].date()} → {rain_ts.index[-1].date()})")
print(f"Monthly temperature series: {len(temp_ts)} observations  ({temp_ts.index[0].date()} → {temp_ts.index[-1].date()})")
rain_ts.head(6)

---
## 4. Visual Exploration

Before any modelling, **always plot your data**. Look for:
- A trend (upward or downward drift over decades)
- Seasonality (same bumps every year — India's monsoon!)
- Outlier years or sudden structural breaks


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

# ── Rainfall ──────────────────────────────────────────────────────────────
axes[0].plot(rain_ts.index, rain_ts.values, color='steelblue', linewidth=0.8, alpha=0.7)
axes[0].plot(rain_ts.rolling(12).mean().index,
             rain_ts.rolling(12).mean().values,
             color='navy', linewidth=2.0, label='12-month rolling mean')
axes[0].set_title('India Monthly Average Rainfall (1901–2024)', fontweight='bold')
axes[0].set_ylabel('Rainfall (mm)')
axes[0].legend()

# ── Temperature ───────────────────────────────────────────────────────────
axes[1].plot(temp_ts.index, temp_ts.values, color='tomato', linewidth=0.8, alpha=0.7)
axes[1].plot(temp_ts.rolling(12).mean().index,
             temp_ts.rolling(12).mean().values,
             color='darkred', linewidth=2.0, label='12-month rolling mean')
axes[1].set_title('India Monthly Average Temperature (1901–2024)', fontweight='bold')
axes[1].set_ylabel('Temperature (°C)')
axes[1].legend()

plt.tight_layout()
plt.show()
print("Observation: Rainfall shows strong seasonal spikes (monsoon). Temperature also has clear seasonality.")

### 4.1 Seasonal Box Plots — The Monsoon Fingerprint

A box plot per month shows the **distribution of rainfall/temperature for each calendar month** across all years.  
This is a quick, powerful way to confirm seasonality and spot outliers.


In [ ]:
# Rebuild with month labels for plotting
rain_df = rain_ts.to_frame()
rain_df['Month'] = rain_df.index.month
rain_df['Month_name'] = rain_df['Month'].map(dict(enumerate(MONTHS, 1)))

temp_df = temp_ts.to_frame()
temp_df['Month'] = temp_df.index.month
temp_df['Month_name'] = temp_df['Month'].map(dict(enumerate(MONTHS, 1)))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.boxplot(data=rain_df, x='Month_name', y='rainfall_mm',
            order=MONTHS, palette='Blues', ax=axes[0])
axes[0].set_title('Monthly Rainfall Distribution (1901–2024)', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Rainfall (mm)')

sns.boxplot(data=temp_df, x='Month_name', y='temp_C',
            order=MONTHS, palette='Reds', ax=axes[1])
axes[1].set_title('Monthly Temperature Distribution (1901–2024)', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Temperature (°C)')

plt.tight_layout()
plt.show()
print("""
Key insight:
  Rainfall peaks June–September (Indian Summer Monsoon) — this is strong SEASONALITY (s=12).
  Temperature peaks in May–June and dips in December–January — a classic bimodal pattern.
""")

### 4.2 Annual Trends — Is India Getting Wetter or Warmer?


In [ ]:
annual_rain = rf_raw.set_index('Year')['Annual'].astype(float)
annual_temp = tmp_raw.set_index('Year')['Annual'].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, series, color, title, unit in [
    (axes[0], annual_rain, 'steelblue', 'Annual Rainfall', 'mm'),
    (axes[1], annual_temp, 'tomato',    'Annual Temperature', '°C'),
]:
    ax.bar(series.index, series.values, color=color, alpha=0.6, width=0.8)
    # 10-year rolling trend
    roll = series.rolling(10, center=True).mean()
    ax.plot(roll.index, roll.values, color='black', linewidth=2, label='10-yr rolling mean')
    ax.set_title(f'India {title} (1901–2024)', fontweight='bold')
    ax.set_ylabel(f'{title} ({unit})')
    ax.set_xlabel('Year')
    ax.legend()

plt.tight_layout()
plt.show()

---
## 5. Seasonal Decomposition

We can formally **split** the series into its four components using `seasonal_decompose`.

- **Additive model**: `Observed = Trend + Seasonal + Residual`  
- **Multiplicative model**: `Observed = Trend × Seasonal × Residual` (better when seasonality scales with the level)

For rainfall, the monsoon amplitude is relatively fixed → **additive** is fine.  
We use a recent 30-year window (1995–2024) for clarity.


In [ ]:
rain_recent = rain_ts['1995':]

decomp = seasonal_decompose(rain_recent, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
pairs = [
    (decomp.observed,  'Observed',  'steelblue'),
    (decomp.trend,     'Trend',     'navy'),
    (decomp.seasonal,  'Seasonal',  'seagreen'),
    (decomp.resid,     'Residual',  'gray'),
]
for ax, (series, label, color) in zip(axes, pairs):
    ax.plot(series, color=color, linewidth=1.2)
    ax.set_ylabel(label, fontweight='bold')
    ax.axhline(0, color='black', linestyle='--', linewidth=0.5)

axes[0].set_title('Additive Decomposition — India Monthly Rainfall (1995–2024)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print("""
Reading the decomposition:
  Trend   → gentle multi-decadal variation; no strong upward/downward drift.
  Seasonal→ the same spike every summer — the monsoon pattern, repeating every 12 months.
  Residual→ random noise after removing trend & seasonality. We want THIS to be white noise.
""")

---
## 6. Stationarity — The Foundation of ARIMA

### Why does ARIMA need stationarity?

ARIMA models assume the **statistical properties are constant in time**.  
A series with a trend or changing variance violates this, making forecasts unreliable.

> *Analogy: You can predict the next number in `3, 3, 3, 3, 3...` easily.  
> You cannot predict `1, 2, 4, 8, 16...` with the same formula applied to raw values.*

### 6.1 Augmented Dickey-Fuller (ADF) Test

| Hypothesis | Meaning |
|---|---|
| **H₀** (null) | The series has a **unit root** → it is **non-stationary** |
| **H₁** (alt)  | The series is **stationary** |

**Rule**: If `p-value < 0.05` → reject H₀ → series is stationary.


In [ ]:
def adf_report(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    stat, pval, lags, nobs, crit, _ = result
    verdict = 'STATIONARY ✓' if pval < 0.05 else 'NON-STATIONARY ✗'
    print(f"""
  ── ADF Test: {name} ──────────────────────────────────
  ADF Statistic : {stat:>10.4f}
  p-value       : {pval:>10.4f}   {'< 0.05 → reject H₀' if pval < 0.05 else '>= 0.05 → fail to reject H₀'}
  Critical (5%) : {crit['5%']:>10.4f}
  Verdict       : {verdict}
""")
    return pval

print("Testing raw monthly series:")
p_rain = adf_report(rain_ts, 'India Monthly Rainfall')
p_temp = adf_report(temp_ts, 'India Monthly Temperature')

### 6.2 Differencing — Making the Series Stationary

**Differencing** = subtract each value from the previous one: `y'_t = y_t - y_{t-1}`

- **1st difference** removes a linear trend → parameter `d=1`
- **Seasonal difference** (lag-12) removes seasonal pattern → parameter `D=1, s=12`

The number of differences needed = the `d` (and `D`) in ARIMA.


In [ ]:
rain_diff1   = rain_ts.diff(1).dropna()          # 1st regular difference
rain_diff_s  = rain_ts.diff(12).dropna()          # seasonal difference (lag-12)
rain_diff_1s = rain_ts.diff(1).diff(12).dropna()  # both combined

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=False)

for ax, series, title, color in [
    (axes[0], rain_ts,       'Original Rainfall', 'steelblue'),
    (axes[1], rain_diff1,    '1st Difference  (d=1)',       'darkorange'),
    (axes[2], rain_diff_s,   'Seasonal Difference (D=1, s=12)',  'seagreen'),
    (axes[3], rain_diff_1s,  'Regular + Seasonal Difference (d=1, D=1)', 'purple'),
]:
    ax.plot(series, color=color, linewidth=0.9)
    ax.axhline(series.mean(), color='black', linestyle='--', linewidth=1.0, label=f'mean={series.mean():.1f}')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('mm')
    ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print("ADF test on differenced series:")
for name, series in [('1st diff', rain_diff1), ('Seasonal diff', rain_diff_s), ('Both diffs', rain_diff_1s)]:
    adf_report(series, name)

---
## 7. ACF & PACF — The Order Identification Tools

Once the series is stationary, we use two correlation plots to figure out `p` and `q`:

| Plot | Full Name | What it tells you |
|------|-----------|--------------------|
| **ACF** | Autocorrelation Function | Correlation between the series and its own past values (lags) |
| **PACF** | Partial Autocorrelation Function | Direct correlation at each lag, removing indirect effects |

### How to read ACF/PACF to choose p and q

```
                ACF behaviour       PACF behaviour
AR(p) model  :  tails off slowly    cuts off after lag p
MA(q) model  :  cuts off after q    tails off slowly
ARMA(p,q)    :  tails off           tails off
```

> **Blue shaded region** = 95% confidence band. Spikes **outside** this band are statistically significant.

For **seasonal** patterns: look for spikes at lag **12, 24, 36** (multiples of s=12).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 9))

plot_acf(rain_ts, lags=48, alpha=0.05, ax=axes[0, 0], title='ACF — Original Rainfall')
plot_pacf(rain_ts, lags=48, alpha=0.05, ax=axes[0, 1], title='PACF — Original Rainfall', method='ywm')

plot_acf(rain_diff_1s, lags=48, alpha=0.05, ax=axes[1, 0], title='ACF — After Differencing (d=1, D=1)')
plot_pacf(rain_diff_1s, lags=48, alpha=0.05, ax=axes[1, 1], title='PACF — After Differencing (d=1, D=1)', method='ywm')

for ax in axes.flatten():
    ax.set_xlabel('Lag (months)')
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

plt.tight_layout()
plt.show()
print("""
Reading the plots:
  Original ACF : Strong spikes at lags 12, 24, 36 → confirms strong s=12 seasonality.
  After diff   : Spikes mostly within bands → series is close to white noise → ready to model!
  
  Non-seasonal order hint: check lags 1–11 in the differenced ACF/PACF.
  Seasonal order hint    : check lags at multiples of 12.
""")

---
## 8. The Box-Jenkins Methodology

Box & Jenkins (1976) defined a classic **3-step cycle** for building ARIMA models:

```
┌─────────────────────────────────────────────────────────┐
│  Step 1: IDENTIFICATION                                 │
│    • Check ACF/PACF → choose tentative p, d, q, P, D, Q│
│    • Use AIC/BIC to compare candidate models            │
└─────────────────┬───────────────────────────────────────┘
                  ↓
┌─────────────────────────────────────────────────────────┐
│  Step 2: ESTIMATION                                     │
│    • Fit the chosen model by Maximum Likelihood         │
│    • Check parameter significance (t-test, p-values)   │
└─────────────────┬───────────────────────────────────────┘
                  ↓
┌─────────────────────────────────────────────────────────┐
│  Step 3: DIAGNOSTIC CHECK                               │
│    • Residuals should be WHITE NOISE (iid, mean=0)      │
│    • ACF of residuals: no significant spikes            │
│    • Ljung-Box test: p-values > 0.05                    │
│                                                         │
│    → If checks fail, go back to Step 1 and try again   │
└─────────────────────────────────────────────────────────┘
```

### ARIMA Notation

| Parameter | Controls | Range typically |
|-----------|----------|----------------|
| **p** | AR order — how many past values are used | 0, 1, 2 |
| **d** | Degree of non-seasonal differencing | 0, 1 |
| **q** | MA order — how many past errors are used | 0, 1, 2 |

### SARIMA Notation: ARIMA(p,d,q)(P,D,Q)ₛ

For data with **seasonal patterns** (our rainfall has s=12), we add a seasonal layer:

| Parameter | Controls |
|-----------|----------|
| **P** | Seasonal AR order |
| **D** | Seasonal differencing degree |
| **Q** | Seasonal MA order |
| **s** | Season length (12 for monthly) |

> *ARIMA(1,1,1)(1,1,1)₁₂ is a very common seasonal model for monthly climate data.*


---
## 9. Step 1 — Model Identification

We use `auto_arima` (pmdarima) to search for the best order automatically using AIC as criterion.  
Think of it as an exhaustive "try all reasonable combinations and pick the best one" search.


In [ ]:
# We model the full rainfall series (1901–2024)
# Hold out the last 12 months as a test set to evaluate forecast accuracy

TRAIN_END = '2023-12'
train = rain_ts[:TRAIN_END]
test  = rain_ts['2024':]

print(f"Training set : {len(train)} months  ({train.index[0].date()} → {train.index[-1].date()})")
print(f"Test set     : {len(test)}  months  ({test.index[0].date()} → {test.index[-1].date()})")

In [ ]:
if HAS_PMDARIMA:
    print("Running auto_arima (this may take ~30 seconds)...")
    auto_model = auto_arima(
        train,
        seasonal=True, m=12,         # s = 12 (monthly seasonality)
        d=None, D=None,              # let auto_arima determine d and D
        max_p=3, max_q=3,
        max_P=2, max_Q=2,
        information_criterion='aic',
        stepwise=True,               # faster stepwise search
        error_action='ignore',
        suppress_warnings=True,
        trace=True                   # show the search progress
    )
    print("\nBest model found:")
    print(auto_model.summary())
    best_order         = auto_model.order
    best_seasonal_order = auto_model.seasonal_order
else:
    # Fallback based on ACF/PACF inspection: a classic SARIMA(1,0,1)(1,1,1)12
    best_order         = (1, 0, 1)
    best_seasonal_order = (1, 1, 1, 12)
    print(f"pmdarima not available. Using fallback: ARIMA{best_order}x{best_seasonal_order}")

---
## 10. Step 2 — Estimation (Model Fitting)

Now we fit the chosen SARIMA model to training data using **statsmodels SARIMAX**.  
The model estimates parameters by maximizing the **log-likelihood** of the data.


In [ ]:
print(f"Fitting SARIMA{best_order}x{best_seasonal_order} ...")

sarima_model = SARIMAX(
    train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

fitted = sarima_model.fit(disp=False)
print(fitted.summary())

### Reading the Summary Table

| Column | Meaning |
|--------|---------|
| **coef** | Estimated parameter value |
| **std err** | Standard error of the estimate |
| **z / P>|z|** | z-statistic and p-value — if p < 0.05, the parameter is significant |
| **AIC / BIC** | Lower is better — use to compare models |

> **Insignificant** parameters (p > 0.05) can sometimes be dropped to simplify the model.


---
## 11. Step 3 — Diagnostic Checks

A good model transforms the data so that its **residuals look like white noise**:  
- Zero mean, constant variance
- No autocorrelation (each residual is independent)
- Approximately normally distributed

We check this three ways:
1. **Residual time plot** — no obvious pattern
2. **ACF of residuals** — all lags inside the confidence band
3. **Ljung-Box test** — formal statistical test for independence


In [ ]:
residuals = fitted.resid

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── 1. Residual time plot ──────────────────────────────────────────────────
axes[0, 0].plot(residuals, color='gray', linewidth=0.8)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuals over Time', fontweight='bold')
axes[0, 0].set_ylabel('Residual (mm)')
axes[0, 0].set_xlabel('Date')

# ── 2. Histogram of residuals ──────────────────────────────────────────────
axes[0, 1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[0, 1].set_title('Histogram of Residuals', fontweight='bold')
axes[0, 1].set_xlabel('Residual (mm)')
axes[0, 1].set_ylabel('Frequency')

# ── 3. ACF of residuals ───────────────────────────────────────────────────
plot_acf(residuals, lags=48, alpha=0.05, ax=axes[1, 0], title='ACF of Residuals (should be white noise)')
axes[1, 0].set_xlabel('Lag (months)')

# ── 4. Q-Q plot (normality) ────────────────────────────────────────────────
from scipy import stats
(osm, osr), (slope, intercept, r) = stats.probplot(residuals.dropna(), dist='norm')
axes[1, 1].scatter(osm, osr, s=5, color='tomato', alpha=0.5)
axes[1, 1].plot(osm, slope * np.array(osm) + intercept, color='black', linewidth=1.5)
axes[1, 1].set_title(f'Q-Q Plot of Residuals  (R²={r**2:.3f})', fontweight='bold')
axes[1, 1].set_xlabel('Theoretical Quantiles')
axes[1, 1].set_ylabel('Sample Quantiles')

plt.suptitle('Model Diagnostic Plots', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Ljung-Box test for residual autocorrelation
lb_result = acorr_ljungbox(residuals.dropna(), lags=[12, 24, 36, 48], return_df=True)

print("Modified Box-Pierce (Ljung-Box) Chi-Square Test:")
print("H₀: residuals are independently distributed (white noise)")
print("H₁: residuals have autocorrelation (model is inadequate)")
print()
print(lb_result.to_string())
print()

if (lb_result['lb_pvalue'] > 0.05).all():
    print("✓  All p-values > 0.05 → Fail to reject H₀ → Residuals are WHITE NOISE → Model is adequate!")
else:
    failed = lb_result[lb_result['lb_pvalue'] <= 0.05]
    print(f"✗  Some p-values ≤ 0.05 at lags: {failed.index.tolist()} → Model may need refinement.")

---
## 12. In-Sample Fit — How Well Did the Model Learn?


In [ ]:
fitted_values = fitted.fittedvalues

# Focus on last 5 years for visual clarity
plot_start = '2018'

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train[plot_start:], label='Actual (train)', color='steelblue', linewidth=1.5)
ax.plot(fitted_values[plot_start:], label='Fitted',  color='orange', linewidth=1.5, linestyle='--')
ax.set_title(f'Actual vs Fitted (SARIMA{best_order}x{best_seasonal_order}) — 2018–2023', fontweight='bold')
ax.set_ylabel('Rainfall (mm)')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.show()

# Error metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
common = train[plot_start:].align(fitted_values[plot_start:], join='inner')
mae  = mean_absolute_error(common[0], common[1])
rmse = np.sqrt(mean_squared_error(common[0], common[1]))
print(f"In-sample (2018–2023)  MAE = {mae:.2f} mm   RMSE = {rmse:.2f} mm")

---
## 13. Forecasting the Next 12 Months

This is the **payoff** — we ask the fitted model to project forward.  
The model provides:
- **Point forecast** — the most likely value for each future month
- **Confidence intervals** — the range within which the true value is likely to fall (95% by default)

> The confidence band widens further into the future — uncertainty grows with forecast horizon.


In [ ]:
# Generate 12-month forecast beyond the training window (Jan 2025 – Dec 2025)
forecast_steps = 12
forecast_result = fitted.get_forecast(steps=forecast_steps)
forecast_mean   = forecast_result.predicted_mean
forecast_ci     = forecast_result.conf_int(alpha=0.05)   # 95% CI

# Label the forecast months clearly
forecast_df = pd.DataFrame({
    'Month':    [f"{d.strftime('%b %Y')}" for d in forecast_mean.index],
    'Forecast': forecast_mean.values.round(2),
    'Lower 95%': forecast_ci.iloc[:, 0].values.round(2),
    'Upper 95%': forecast_ci.iloc[:, 1].values.round(2),
})

print(f"12-Month Rainfall Forecast  (SARIMA{best_order}x{best_seasonal_order})\n")
print(forecast_df.to_string(index=False))
print(f"\nAverage forecast : {forecast_df['Forecast'].mean():.1f} mm/month")

In [ ]:
# ── Main forecast plot ─────────────────────────────────────────────────────
plot_from = '2020'
fig, ax = plt.subplots(figsize=(16, 6))

# Historical
ax.plot(train[plot_from:], color='steelblue', linewidth=1.5, label='Historical (train)')

# 2024 actuals (held-out test set)
if len(test) > 0:
    ax.plot(test, color='green', linewidth=1.8, linestyle='-', marker='o', markersize=4, label='Actual 2024')

# Forecast
ax.plot(forecast_mean.index, forecast_mean.values,
        color='tomato', linewidth=2.5, linestyle='--', marker='s', markersize=5, label='Forecast (12 months)')

# Confidence interval shading
ax.fill_between(
    forecast_mean.index,
    forecast_ci.iloc[:, 0].clip(lower=0),   # rainfall can't be negative
    forecast_ci.iloc[:, 1],
    color='tomato', alpha=0.15, label='95% Confidence Interval'
)

# Vertical separator
ax.axvline(train.index[-1], color='gray', linestyle=':', linewidth=1.5, label='Forecast start')

ax.set_title(f'India Monthly Rainfall — SARIMA{best_order}x{best_seasonal_order} Forecast', fontweight='bold', fontsize=14)
ax.set_ylabel('Rainfall (mm)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Zoom: forecast only — month-by-month bar chart ────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

bars = ax.bar(
    forecast_df['Month'], forecast_df['Forecast'],
    color=['steelblue' if m in ['Jun','Jul','Aug','Sep'] else 'lightsteelblue'
           for m in [d.strftime('%b') for d in forecast_mean.index]],
    edgecolor='white', zorder=3
)

# Error bars for CI
lower_err = forecast_df['Forecast'] - forecast_df['Lower 95%'].clip(lower=0)
upper_err = forecast_df['Upper 95%'] - forecast_df['Forecast']
ax.errorbar(
    range(len(forecast_df)), forecast_df['Forecast'],
    yerr=[lower_err, upper_err],
    fmt='none', color='black', capsize=5, linewidth=1.5
)

# Value labels
for bar, val in zip(bars, forecast_df['Forecast']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('12-Month Rainfall Forecast — Month-by-Month (darker = monsoon months)', fontweight='bold')
ax.set_ylabel('Rainfall (mm)')
ax.set_xlabel('Month')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## 14. Forecast Evaluation (Backtesting on 2024)

We held out 2024 data. Let's see how close the model came!


In [ ]:
if len(test) > 0:
    n_eval = min(len(test), forecast_steps)
    actual_eval   = test.values[:n_eval]
    forecast_eval = forecast_mean.values[:n_eval]

    mae_test  = mean_absolute_error(actual_eval, forecast_eval)
    rmse_test = np.sqrt(mean_squared_error(actual_eval, forecast_eval))
    mape_test = np.mean(np.abs((actual_eval - forecast_eval) / (actual_eval + 1e-6))) * 100

    eval_df = pd.DataFrame({
        'Month':    forecast_df['Month'][:n_eval].values,
        'Actual':   actual_eval.round(1),
        'Forecast': forecast_eval.round(1),
        'Error':    (actual_eval - forecast_eval).round(1),
    })

    print("Forecast vs Actual — 2024 Holdout Evaluation\n")
    print(eval_df.to_string(index=False))
    print(f"""
  ── Error Metrics ──────────────────────────
  MAE  = {mae_test:.2f} mm   (average absolute error)
  RMSE = {rmse_test:.2f} mm   (penalises large errors more)
  MAPE = {mape_test:.1f} %    (average % error)
""")
else:
    print("No held-out test data available for evaluation.")

---
## 15. Bonus — Temperature SARIMA Forecast

Let's quickly apply the same workflow to temperature data.


In [ ]:
print("ADF test on temperature series:")
adf_report(temp_ts, 'Monthly Temperature')

temp_train = temp_ts[:'2023']
temp_test  = temp_ts['2024':]

# Temperature is generally stationary but seasonal → SARIMA(1,0,1)(1,1,1)12
print("Fitting SARIMA(1,0,1)(1,1,1)12 on temperature...")
temp_model = SARIMAX(
    temp_train,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

temp_forecast = temp_model.get_forecast(steps=12)
temp_fc_mean  = temp_forecast.predicted_mean
temp_fc_ci    = temp_forecast.conf_int(alpha=0.05)

print("\n12-Month Temperature Forecast:")
temp_fc_df = pd.DataFrame({
    'Month': [d.strftime('%b %Y') for d in temp_fc_mean.index],
    'Forecast (°C)': temp_fc_mean.values.round(2),
    'Lower 95%': temp_fc_ci.iloc[:, 0].values.round(2),
    'Upper 95%': temp_fc_ci.iloc[:, 1].values.round(2),
})
print(temp_fc_df.to_string(index=False))

In [ ]:
plot_from_t = '2020'
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(temp_train[plot_from_t:], color='tomato', linewidth=1.5, label='Historical Temperature')
if len(temp_test) > 0:
    ax.plot(temp_test, color='green', linewidth=1.8, marker='o', markersize=4, label='Actual 2024')
ax.plot(temp_fc_mean, color='darkred', linewidth=2.5, linestyle='--', marker='s', markersize=5, label='Forecast (12 months)')
ax.fill_between(temp_fc_mean.index, temp_fc_ci.iloc[:, 0], temp_fc_ci.iloc[:, 1],
                color='darkred', alpha=0.12, label='95% CI')
ax.axvline(temp_train.index[-1], color='gray', linestyle=':', linewidth=1.5)

ax.set_title('India Monthly Temperature — SARIMA(1,0,1)(1,1,1)₁₂ Forecast', fontweight='bold', fontsize=14)
ax.set_ylabel('Temperature (°C)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 16. Summary & Cheat Sheet

### The ARIMA Roadmap

```
Raw data
  │
  ├─ Plot it! Look for trend & seasonality
  │
  ├─ ADF test
  │     └─ not stationary? → difference (d++, D++) → re-test
  │
  ├─ ACF / PACF on stationary series
  │     ├─ PACF cuts off at p  → AR(p)
  │     └─ ACF  cuts off at q  → MA(q)
  │
  ├─ Fit ARIMA(p,d,q)(P,D,Q)₁₂ with statsmodels
  │
  ├─ Diagnose residuals
  │     ├─ ACF of residuals: all inside band?
  │     └─ Ljung-Box p > 0.05 at all lags?
  │
  └─ Forecast! (point forecast + confidence intervals)
```

### Quick-Reference: Choosing p, d, q

| Symptom | Action |
|---------|--------|
| ACF decays slowly | Series is non-stationary → difference (d+1) |
| ACF cuts off at lag q | Use MA(q) |
| PACF cuts off at lag p | Use AR(p) |
| Spikes at lag 12, 24 in ACF | Add seasonal component (s=12, D=1) |
| Ljung-Box p < 0.05 | Model is missing structure → try larger p or q |

### Interpreting Confidence Intervals

- **Narrow CI** = model is confident (good historical patterns, short horizon)
- **Wide CI** = high uncertainty (long horizon, noisy series)
- Always clip lower CI at 0 for non-negative quantities like rainfall

### Limitations of ARIMA

1. Assumes **linear** relationships — won't capture complex climate-ocean couplings (ENSO etc.)
2. Assumes **stationarity** — climate change may cause non-stationarity over long periods
3. **Short-term forecasts** are more reliable than long-term ones
4. External variables (temperature, ENSO index) can be incorporated via **SARIMAX** (the X = eXogenous inputs)

---
### Further Reading
- Box, G.E.P. & Jenkins, G.M. (1976). *Time Series Analysis: Forecasting and Control*. Holden-Day.
- Sharma, M.K. et al. (2020). *Time series analysis on precipitation with missing data using stochastic SARIMA*. MAUSAM, 71(4), 617–624.
- statsmodels docs: https://www.statsmodels.org/stable/statespace.html
- pmdarima docs: https://alkaline-ml.com/pmdarima/
